# 01 — Data Curation: ChEMBL EGFR Bioactivity Data

This notebook walks through fetching, cleaning, and curating EGFR (ChEMBL target **CHEMBL203**) bioactivity data.

**Steps:**
1. Query ChEMBL API for IC50 data
2. Clean and preprocess the data
3. Compute pIC50 and label compounds
4. Assess data quality
5. Lipinski Rule of 5 analysis

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from rdkit import Chem
from rdkit.Chem import Draw

from src.components.data_loader import ChEMBLDataLoader

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Configuration

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

loader = ChEMBLDataLoader(config)
print(f"Target: {config['data']['chembl_target_id']}")
print(f"Bioactivity type: {config['data']['bioactivity_type']}")

## 2. Fetch Raw Bioactivity Data from ChEMBL

In [ ]:
raw_df = loader.fetch_bioactivity_data()
loader.save_raw_data(raw_df)
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Data quality overview
print("Missing values per column:")
print(raw_df.isnull().sum())
print(f"\nDuplicate molecule IDs: {raw_df['molecule_chembl_id'].duplicated().sum()}")

## 3. Preprocess and Curate

In [ ]:
curated_df = loader.preprocess_bioactivity(raw_df)
loader.save_curated_data(curated_df)
print(f"Curated dataset shape: {curated_df.shape}")
curated_df.head()

## 4. pIC50 Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# pIC50 distribution
sns.histplot(curated_df["pIC50"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].axvline(x=6.0, color="red", linestyle="--", label="Active threshold (pIC50=6)")
axes[0].set_xlabel("pIC50")
axes[0].set_ylabel("Count")
axes[0].set_title("pIC50 Distribution")
axes[0].legend()

# Activity class balance
class_counts = curated_df["activity_class"].value_counts()
axes[1].bar(class_counts.index, class_counts.values, color=["#2ecc71", "#e74c3c"])
axes[1].set_xlabel("Activity Class")
axes[1].set_ylabel("Count")
axes[1].set_title("Active vs Inactive Compounds")
for i, v in enumerate(class_counts.values):
    axes[1].text(i, v + 10, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../results/plots/pIC50_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Lipinski Rule of 5 Analysis

In [ ]:
lipinski_data = curated_df["canonical_smiles"].apply(loader.compute_lipinski)
lipinski_df = pd.DataFrame(lipinski_data.tolist())
curated_with_lipinski = pd.concat([curated_df, lipinski_df], axis=1)

pass_rate = lipinski_df["Lipinski_Pass"].mean() * 100
print(f"Lipinski Rule of 5 pass rate: {pass_rate:.1f}%")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
props = [("MolWt", 500, "Molecular Weight"), ("LogP", 5, "LogP"),
         ("NumHDonors", 5, "H-Bond Donors"), ("NumHAcceptors", 10, "H-Bond Acceptors")]

for ax, (col, threshold, title) in zip(axes.flat, props):
    sns.histplot(curated_with_lipinski[col], bins=40, ax=ax, color="steelblue", kde=True)
    ax.axvline(x=threshold, color="red", linestyle="--", label=f"Ro5 limit ({threshold})")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig("../results/plots/lipinski_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Summary Statistics

In [ ]:
print("=== Dataset Summary ===")
print(f"Total compounds: {len(curated_df)}")
print(f"Active (pIC50 >= 6): {(curated_df['activity_class'] == 'active').sum()}")
print(f"Inactive (pIC50 < 6): {(curated_df['activity_class'] == 'inactive').sum()}")
print(f"\npIC50 statistics:")
print(curated_df["pIC50"].describe())